In [1]:
import torch
from transformers import AutoModelForCausalLM , AutoTokenizer
from datasets import load_dataset



In [2]:
from huggingface_hub import login
login()

In [3]:
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b",
    dtype=torch.float16,
    device_map="cuda"
)

print(model)

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): GemmaTextScaledWordEmbedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Gem

In [4]:
model_len = len(model.model.layers)

In [5]:
model.model.layers[model_len//2]

GemmaDecoderLayer(
  (self_attn): GemmaAttention(
    (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): GemmaMLP(
    (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
    (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
    (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
    (act_fn): GELUActivation()
  )
  (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
  (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
)

In [6]:
model.config

GemmaConfig {
  "architectures": [
    "GemmaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 2,
  "dtype": "float16",
  "eos_token_id": 1,
  "head_dim": 256,
  "hidden_act": "gelu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 16384,
  "max_position_embeddings": 8192,
  "model_type": "gemma",
  "num_attention_heads": 8,
  "num_hidden_layers": 18,
  "num_key_value_heads": 1,
  "pad_token_id": 0,
  "rms_norm_eps": 1e-06,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.13.1",
  "use_bidirectional_attention": null,
  "use_cache": true,
  "vocab_size": 256000
}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2")

In [8]:
class StopForwad(Exception): pass

acts = {}
def hook(module , inp , out):
    print("Hook active")
    acts["vector"] = out[0]
    raise StopForwad


In [9]:
device = "cuda"

In [10]:


token = tokenizer("I'm a language model")
x =  torch.tensor(token.input_ids).unsqueeze(0).to(device)

In [11]:
h= model.model.layers[model_len//2].register_forward_hook(hook)

try:
    model(x)
except StopForwad:
    pass

h.remove()

Hook active


In [12]:
v =  acts["vector"] 

In [13]:
v.shape

torch.Size([7, 2048])

In [14]:
v[:1].shape

torch.Size([1, 2048])

In [15]:
from model import JumpRELUSAE

In [16]:
m = JumpRELUSAE(d_sea = 2**12)

sum(p.numel() for p in m.parameters())

18884864

In [7]:
data = load_dataset("HuggingFaceFW/fineweb-edu" ,streaming=True , split="train")
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

In [9]:
device = "cuda"

data = data.take(2)
for text in data:
    tokens = tokenizer(text["text"], return_tensors = "pt").input_ids
    x = tokens.to(device)
    print(x.shape)

torch.Size([1, 752])
torch.Size([1, 213])
